# 05 — Neural Network Implied Volatility Surface

This notebook trains a feedforward neural network to learn the mapping
$(\ln(K/S),\, T) \to \sigma_{IV}$ from real SPY call options data, then
evaluates it against the bicubic spline already implemented in
`pricer.implied_vol.fit_surface`.

**Sections**
1. [Setup and data loading](#1-setup)
2. [Feature engineering and walk-forward split](#2-features)
3. [Model architecture](#3-architecture)
4. [Training](#4-training)
5. [Evaluation: metrics and residual plots](#5-evaluation)
6. [Surface comparison: NN vs spline](#6-surfaces)
7. [Discussion](#7-discussion)

## 1. Setup <a id='1-setup'></a>

In [ ]:
from __future__ import annotations

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from pricer.implied_vol import fit_surface
from pricer.vol_surface_nn import (
    FEATURE_COLS,
    load_and_prepare,
    walk_forward_split,
    VolSurfaceNet,
    VolSurfaceTrainer,
    evaluate,
    plot_evaluation,
    plot_loss_curves,
    predict_surface,
    plot_surfaces,
)

print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 2. Feature engineering and walk-forward split <a id='2-features'></a>

### Why log-moneyness and not raw strike?

Raw strikes are scale-dependent — SPY sits around 500, AAPL around 200.
Log-moneyness $m = \ln(K/S)$ is dimensionless: $m=0$ is ATM regardless of
the underlying price, and symmetric wings have equal magnitude.
This also makes the feature range compact and consistent across tickers,
which is essential for normalisation to work well.

### Why walk-forward and not random split?

Options data is time-ordered. A random 80/20 split could train on March
expiries and test on February expiries — the model would have seen the
future during training. Walk-forward keeps test data strictly in the
future relative to training data, matching real deployment conditions.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

TICKER = 'SPY'
CHAIN_PATH = f'data/{TICKER.lower()}_chain.csv'

if not Path(CHAIN_PATH).exists():
    print(f'{CHAIN_PATH} not found — generating synthetic BS data for CI.')
    from pricer.black_scholes import price as _bs
    from pricer.models import OptionParams
    Path('data').mkdir(exist_ok=True)
    S, r, q = 550.0, 0.05, 0.01
    expiry_dates = pd.date_range(
        start=pd.Timestamp.today().normalize() + pd.Timedelta(days=30),
        periods=12, freq='ME',
    )
    rows = []
    for exp in expiry_dates:
        T = max((exp - pd.Timestamp.today().normalize()).days / 365.25, 0.01)
        for K in np.linspace(S * 0.85, S * 1.15, 12):
            m = np.log(K / S)
            sigma = 0.20 + 0.08 * m**2 - 0.04 * m
            p = OptionParams(S=S, K=K, T=T, r=r, sigma=sigma, q=q)
            mid = _bs(p, 'call')
            rows.append({
                'ticker': TICKER, 'kind': 'call', 'strike': round(K, 2),
                'expiry': round(T, 6), 'expiry_date': exp.date(),
                'mid': round(mid, 4), 'bid': round(mid * 0.99, 4),
                'ask': round(mid * 1.01, 4), 'last_price': round(mid, 4),
                'volume': 1000, 'open_interest': 10000,
                'implied_volatility_yf': sigma, 'S': S, 'r': r, 'q': q,
            })
    pd.DataFrame(rows).to_csv(CHAIN_PATH, index=False)
    print(f'Generated {len(rows)} synthetic contracts.')

df = load_and_prepare(CHAIN_PATH, kind='call', use_rates=False)
print(f'Contracts after filtering: {len(df)}')
print(f'Expiry dates: {df["expiry_date"].nunique()}')
df.describe()

In [ ]:
train, val, test = walk_forward_split(df, test_frac=0.15, val_frac=0.15)
print(f'Train: {len(train):>5} rows  |  expiry range: {train["expiry_date"].min().date()} → {train["expiry_date"].max().date()}')
print(f'Val:   {len(val):>5} rows  |  expiry range: {val["expiry_date"].min().date()} → {val["expiry_date"].max().date()}')
print(f'Test:  {len(test):>5} rows  |  expiry range: {test["expiry_date"].min().date()} → {test["expiry_date"].max().date()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for split, label, colour in [(train, 'Train', 'steelblue'), (val, 'Val', 'orange'), (test, 'Test', 'crimson')]:
    axes[0].scatter(split['log_moneyness'], split['iv'], s=4, alpha=0.3, c=colour, label=label)
    axes[1].scatter(split['T'], split['iv'], s=4, alpha=0.3, c=colour, label=label)

axes[0].set(xlabel='Log-moneyness ln(K/S)', ylabel='Implied Vol', title='IV vs Moneyness')
axes[1].set(xlabel='T (years)', ylabel='Implied Vol', title='IV vs Expiry')
axes[0].legend(markerscale=3)
fig.tight_layout()
plt.show()

## 3. Model architecture <a id='3-architecture'></a>

The network is a simple feedforward MLP:

$$
\mathbf{h}^{(l)} = \text{Dropout}\!\left(\text{ReLU}\!\left(W^{(l)}\mathbf{h}^{(l-1)} + b^{(l)}\right)\right)
$$

with a scalar linear output (no activation — the target is normalised
via StandardScaler so it can be negative).

**Why ReLU?** It doesn't saturate for large inputs (unlike tanh/sigmoid)
and produces sparse activations that act as implicit regularisation.

**Why Dropout?** During each forward pass, each neuron is zeroed with
probability $p$. This prevents co-adaptation — neurons can't learn to
rely on specific partners, so the network is forced to learn more
distributed representations. At inference `model.eval()` disables it
and implicitly scales weights by $(1-p)^{-1}$.

**Bias-variance trade-off:** wider/deeper nets have more capacity
(lower training loss) but overfit more without regularisation.
The default `[64, 64]` is a reasonable starting point for a few
thousand SPY contracts — increase to `[128, 128, 64]` if the
validation loss plateaus well above the training loss.

In [ ]:
model = VolSurfaceNet(
    input_dim=len(FEATURE_COLS),
    hidden_sizes=[64, 64],
    dropout_rate=0.1,
)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTotal trainable parameters: {n_params:,}')

## 4. Training <a id='4-training'></a>

### Normalisation

`VolSurfaceTrainer` fits a `StandardScaler` on the training set for both
inputs and targets. The scaler subtracts the training mean and divides
by the training standard deviation:

$$\tilde{x} = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

The same fitted scaler (same $\mu$, same $\sigma$) is applied to all
subsequent data — val, test, and live inference. The network therefore
always receives inputs in $\approx [-3, 3]$ and predicts normalised
targets; `trainer.predict()` automatically inverse-transforms back to
real vol units.

If you refit the scaler on test data you leak test statistics into the
model's effective calibration — don't do it.

### Adam + ReduceLROnPlateau

Adam maintains per-parameter adaptive learning rates. When the val loss
stops improving for `patience // 3` epochs, the scheduler halves the
learning rate, allowing the optimiser to take finer steps around the
current minimum. Early stopping then halts training when no improvement
is seen for `patience` epochs and restores the best-seen weights.

In [ ]:
trainer = VolSurfaceTrainer(
    model=model,
    lr=1e-3,
    patience=30,
    max_epochs=500,
    batch_size=256,
)

history = trainer.fit(train, val, feature_cols=FEATURE_COLS)
print(f'Trained for {len(history["train_loss"])} epochs')
print(f'Final train loss: {history["train_loss"][-1]:.6f}')
print(f'Best val loss:    {min(history["val_loss"]):.6f}')

In [ ]:
fig = plot_loss_curves(history)
plt.show()

## 5. Evaluation <a id='5-evaluation'></a>

All metrics are in **vol-point units** (e.g. 0.01 = 1 vol point),
computed on the inverse-transformed predictions.

To build a spline for comparison we need to wrap `fit_surface` so it
accepts `(log_moneyness, T)` rather than `(strike, expiry)`:

In [ ]:
# fit_surface() takes the raw chain DataFrame (with strike, expiry, mid, kind, S, r, q)
# and returns (enriched_df, interpolator) where interpolator(K, T) -> iv.
# We load the raw CSV, restrict to training expiry dates, then wrap the interpolator
# to accept (log_moneyness, T) so it matches the NN's feature space.

raw_chain = pd.read_csv(CHAIN_PATH, parse_dates=["expiry_date"])
raw_chain["expiry_date"] = pd.to_datetime(raw_chain["expiry_date"]).dt.normalize()

# Get the training expiry dates from the prepared split
train_expiry_dates = set(train["expiry_date"])
raw_train = raw_chain[
    (raw_chain["kind"] == "call") &
    (raw_chain["expiry_date"].isin(train_expiry_dates))
].copy()

_, interpolator = fit_surface(raw_train)

# Use the median spot price from training data as the conversion anchor
S_ref = float(raw_train["S"].median())

def spline_fn(m_arr, T_arr):
    """Wrap fit_surface interpolator to accept (log_moneyness, T) inputs."""
    m = np.asarray(m_arr)
    T = np.asarray(T_arr)
    K = S_ref * np.exp(m.ravel())
    T_flat = T.ravel()
    return interpolator(K, T_flat).reshape(m.shape)

print(f"Spline fitted on {len(raw_train)} contracts, S_ref={S_ref:.2f}")

In [ ]:
metrics = evaluate(trainer, test, feature_cols=FEATURE_COLS, spline_fn=spline_fn)

print(f'{'Model':<12}  {'RMSE':>8}  {'MAE':>8}  {'Max AE':>8}')
print('-' * 42)
for name, m in metrics.items():
    print(f'{name:<12}  {m["rmse"]:>8.4f}  {m["mae"]:>8.4f}  {m["max_ae"]:>8.4f}')

In [ ]:
fig = plot_evaluation(trainer, test, feature_cols=FEATURE_COLS, spline_fn=spline_fn)
plt.show()

**Reading the residual plots:**

- **Residuals vs moneyness** — a U-shape here means the model is
  underestimating the smile curvature (underfitting the wings).
  Increase network width or add a third hidden layer.
- **Residuals vs T** — a slope here means the model is systematically
  wrong on short-dated or long-dated options.  Short-dated options
  ($T < 0.1$) often have the biggest residuals because their IVs
  are most sensitive to small price changes.

## 6. Surface comparison: NN vs spline <a id='6-surfaces'></a>

We evaluate both models on a regular grid to visualise the full surface.

In [ ]:
nn_result = predict_surface(
    trainer,
    feature_cols=FEATURE_COLS,
    moneyness_range=(-0.3, 0.3),
    T_range=(0.05, 2.0),
    n_points=60,
)
M, T_grid, IV_nn = nn_result

fig = plot_surfaces(nn_result, spline_fn=spline_fn)
plt.show()

In [ ]:
# Difference surface: NN minus spline
IV_spline = spline_fn(M, T_grid)
IV_diff = IV_nn - IV_spline

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(M, T_grid, IV_diff, cmap='RdBu_r', alpha=0.85, edgecolor='none')
fig.colorbar(surf, ax=ax, shrink=0.5, label='NN − Spline (vol pts)')
ax.set_xlabel('ln(K/S)')
ax.set_ylabel('T (years)')
ax.set_zlabel('IV difference')
ax.set_title('Difference Surface: NN − Spline')
plt.show()

## 7. Discussion <a id='7-discussion'></a>

### Where the NN tends to outperform the spline

**Sparse expiry buckets.** The spline requires a populated grid — if
a particular strike/expiry combination has few contracts, the bicubic
polynomial can oscillate or spike. The NN sees all contracts jointly
and learns a smooth global shape that doesn't degrade as badly in
sparse regions.

**Extrapolation near edges.** At very short expiry ($T < 0.1$) or far
wings ($m < -0.4$), the spline frequently produces extreme values or
even negative vols because it's forced to fit a polynomial through a
small number of noisy points. The NN is constrained by its architecture
to produce smoother, more conservative extrapolations.

**Generalisation across strikes.** The NN has seen all strikes
simultaneously so it implicitly learns the smile shape. The spline
treats each grid cell independently.

### Where the spline tends to outperform the NN

**Dense, clean data regions.** In the liquid core of the surface
($m \in [-0.15, 0.15]$, $T \in [0.1, 1.0]$) the spline interpolates
exactly through every observed point. The NN smooths over them,
introducing a small systematic underfit that shows up as non-zero
residuals even at well-covered grid points.

**No-arbitrage.** The spline makes no claims about arbitrage, but
because it interpolates exactly it at least avoids obvious violations
at observed strikes. The NN can, in principle, produce surfaces that
violate calendar spread or butterfly arbitrage conditions, especially
in sparse regions. Enforcing no-arbitrage would require soft constraints
in the loss function (e.g. a penalty on negative butterfly spreads),
which is beyond the scope of this notebook.

**Sample efficiency.** The spline needs no training data in the ML
sense — it fits instantly on whatever data you give it. The NN needs
enough contracts across enough expiry dates for the gradient descent
to converge to a reasonable function. With fewer than ~500 training
contracts the spline will usually win on RMSE.

### Practical recommendation

For a production vol surface the two approaches are complementary.
Use the spline for liquid, data-rich regions where exact interpolation
is valuable; use the NN (or a parametric model like SVI) for sparse
regions and for smooth extrapolation. A natural extension is to use
the NN residuals as a diagnostic: where the NN and spline disagree
significantly, the data is likely sparse or noisy — flag those
regions for closer inspection.